# 🔬 level_flow — diagnostics: does the procedure survive unseen data?

[`level_flow.ipynb`](./level_flow.ipynb) decides the polarity on the naked signal, optimizes on the training panel and saves the admissible Pareto front as candidates. This notebook takes **any set of those candidates** — plus, if you like, what `config.yaml` ships — and puts them through every check that decides whether a parameter set goes to production, **side by side**. Its centrepiece is §3, the **re-tuning procedure as a tested object**: rolling walk-forward folds that re-optimize only the regime-sensitive knobs on the four months before each test window, scored against two controls — the parameters **frozen** from the first fold and the shipped **defaults** — and stitched into one out-of-sample path. If re-tuning does not beat the frozen control, the strategy ships frozen.

**How to use it.** In §0 set `CANDIDATES` (`"pick"`, trial numbers from the saved front, or `"config"`) and `FOCUS` (the candidate the trade-anatomy figures draw), then Run All.

| § | Question | Candidates |
|---|---|---|
| 2 Stress in-sample | Costs, the 10/3 bps pass bar, pessimistic fills, parameter stability, month by month, trade anatomy | all / FOCUS |
| 3 Walk-forward | Does *re-tune, then trade* beat *freeze* and *never tune* on windows nobody fitted? | procedure |
| 4 Holdout | May–June 2026, opened once, warm; at default and stressed costs | all |
| 5 Verdict | One pass/fail table | all |
| 6 Promote | Write a passing candidate into `config.yaml` | one |

**The gates** (thresholds in §0):

| Gate | Pass when |
|---|---|
| Alpha, train | `alpha_cagr` > 0 warm from `optuna.score_from` |
| Alpha, holdout | `alpha_cagr` > 0 (scored warm) |
| Walk-forward (procedure) | refit alpha > 0 in ≥ 3 of 4 folds and mean > 0 — the same for every candidate |
| Pass bar | holdout alpha > 0 at 10 bps spread / 3 bps slippage — **hard** |
| Enough evidence | ≥ 30 holdout trades |
| Not a narrow peak | no parameter-stability `spike` |
| Re-tuning earns its cost | stitched refit path ≥ stitched frozen path — a warning |
| Beats never tuning | stitched refit path > stitched defaults path — a warning |
| Robust to execution | holdout alpha under pessimistic fills — reported |
| Risk | holdout volatility ≤ buy-and-hold's — a warning unless made hard |

Every window is scored **warm** (`score_from`): the strategy runs on the whole history and only the window's bars count. **Do not tune against §3–§4**: a candidate chosen because it won the holdout has turned the holdout into training data.

## 0 · Choose the candidates

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from scipy import stats

from trading_simple import backtest as bt
from trading_simple import plots, strategies
from trading_simple import optimize as opt
from trading_simple.config import load_settings
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple.db import make_engine

# --- what to diagnose ------------------------------------------------------------------------
CANDIDATES = ["pick"]      # "pick", trial numbers from the saved front (e.g. 412), "config"
FOCUS = None               # the candidate the trade-anatomy figures draw (None = the first one)
STUDY = None               # None = config.yaml's optuna.study

# --- the gates (§5), and effort ---------------------------------------------------------------
GATES = {
    "wf_min_positive_folds": 3,     # of 4 walk-forward folds with positive OOS alpha (refit)
    "pass_bar_bps": (10.0, 3.0),    # (spread, slippage) the holdout must survive — hard
    "min_holdout_trades": 30,
    "vol_is_hard": False,           # False: holdout volatility above buy-and-hold only WARNs
    "pessimistic_min_alpha": None,  # e.g. 0.0 to require alpha > 0 under worst-side fills
}
HOLDOUT_START = "2026-05-01"
HOLDOUT_END = "2026-06-30"    # the last session the WHOLE universe has 1m bars for; a lone
                              # name with later bars would otherwise stretch the holdout
WF_TRIALS = 300            # trials per walk-forward fold study (journal-backed: tops up on re-run)
COST_MULTIPLES = (1.0, 2.0, 3.0)
# Rolling folds: (train start, train end, test start, test end). Four months of training,
# six weeks of test, all inside the train panel; every fold trains and tests WARM.
FOLDS = [("2025-07-01", "2025-10-31", "2025-11-01", "2025-12-15"),
         ("2025-08-16", "2025-12-15", "2025-12-16", "2026-01-31"),
         ("2025-10-01", "2026-01-31", "2026-02-01", "2026-03-15"),
         ("2025-11-16", "2026-03-15", "2026-03-16", "2026-04-30")]
# Structural knobs stay at the candidate's values in every fold; everything else searched is
# re-tuned per fold (the "regime-sensitive" set: tolerances, stop, target, horizon, seats...).
STRUCTURAL = ("tf_levels", "signal_tf", "trend_tf", "trend_gate", "pivot_k", "bin_pct",
              "polarity", "direction", "rank_by", "regime", "scan_minute", "vwap_gate")

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("level_flow")
d = strat.data
STUDY = STUDY or strat.optuna["study"]
SPACE = strat.optuna["space"]
SPACE_INT = {k for k, spec in SPACE.items() if spec["type"] == "int"}
SCORE_FROM = strat.optuna.get("score_from")
RETUNE = [k for k in SPACE if k not in STRUCTURAL]
INIT_CASH = settings.backtest.init_cash
PCT = "{:+.1%}"

saved = yaml.safe_load(opt.study_path(strat, STUDY).with_suffix(".candidates.yaml").read_text())
by_trial = {row["trial"]: row for row in saved["front"]}
FLOOR = saved["pick_min_trades_per_year"]


def resolve(name):
    """A candidate name -> (label, searched params, saved TRAIN metrics or {})."""
    if name == "config":
        return "config", {k: strat.params[k] for k in SPACE}, {}
    trial = saved["pick"] if name == "pick" else int(name)
    if trial not in by_trial:
        raise KeyError(f"trial {trial} is not on the saved front of {STUDY}; "
                       f"choose from {sorted(by_trial)}")
    row = by_trial[trial]
    label = f"pick #{trial}" if trial == saved["pick"] else f"#{trial}"
    return label, row["params"], {k: v for k, v in row.items() if k not in ("trial", "params")}


resolved = [resolve(c) for c in CANDIDATES]
cands = {label: params for label, params, _ in resolved}
FOCUS = resolve(FOCUS)[0] if FOCUS is not None else next(iter(cands))
if FOCUS not in cands:
    raise KeyError(f"FOCUS {FOCUS!r} must be one of CANDIDATES {list(cands)}")
print(f"study {STUDY} · candidates saved {saved['saved']} · {len(by_trial)} on the saved front "
      f"· pick #{saved['pick']} · FOCUS = {FOCUS}")
print(f"re-tuned per fold: {RETUNE}")

params_table = pd.DataFrame(cands)
params_table.insert(0, "config.yaml", [strat.params[k] for k in params_table.index])
display(params_table.style.format(lambda v: f"{v:g}" if isinstance(v, float) else str(v)).apply(
    lambda r: ["" if c == "config.yaml" or r[c] == r["config.yaml"] else "background: #fde3c0"
               for c in r.index], axis=1))
print("shaded = differs from what config.yaml ships")
if trained := {label: m for label, _, m in resolved if m}:
    print("\nsaved TRAIN metrics (level_flow.ipynb §7):")
    display(pd.DataFrame(trained).style.format("{:.3f}"))

## 1 · Load the panels

Two loads: the **train** panel exactly as the optimizer saw it, and the **full-history** panel for every out-of-sample score. `score()` is the one function every later section reads its numbers from: it runs the strategy **warm** (signals on the whole panel, book and benchmark from `start`) and returns the engine's stats — including `alpha_cagr` and `beta` — plus buy-and-hold's own CAGR and volatility.

In [ ]:
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))


def load(end):
    """One panel from data.start to `end` (derived bars if the config asks for them)."""
    return load_panel(engine, syms, d["bar"], d["start"], end,
                      session=d.get("session", "regular"), exchanges=settings.exchanges,
                      resample=d.get("resample"))


def slice_panel(p, start, end):
    """A copy of the panel restricted to [start, end] (end inclusive, whole days)."""
    m = ((p.close.index >= pd.Timestamp(start, tz="UTC"))
         & (p.close.index < pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)))
    idx = p.close.index[m]
    return replace(p, close=p.close.loc[idx], open_=p.open_.loc[idx], high=p.high.loc[idx],
                   low=p.low.loc[idx], volume=p.volume.loc[idx])


def score(params, p, start=None, strategy=None, cfg=None):
    """Every number the gates read, for one parameter set on one window, scored WARM."""
    strategy, cfg = strategy or strat, cfg or settings
    pf, w, fees, slip = bt._run_scored(strategy, p, params, cfg, start)
    out = bt.portfolio_stats(pf)
    out |= bt.active_stats(pf, w, fees=fees, slippage=slip, total_return=out["total_return"],
                           fill=strategy.fill)
    bench_ret, bench = bt.benchmark(w, fees=fees, slippage=slip, fill=strategy.fill)
    bars_per_year = len(w.close.index) / out["years"]
    out["bench_cagr"] = bt._cagr(bench_ret, out["years"])
    out["bench_volatility"] = float(bench.std() * np.sqrt(bars_per_year))
    out["returns"] = bt.equity_curve(pf)["value"].pct_change().fillna(0.0)
    out["bench_returns"], out["pf"], out["window"] = bench, pf, w
    return out


def bench_curve(p, start):
    """Cumulative return of equal-weight buy-and-hold from `start`, for overlays."""
    pf = bt.run_strategy(strategies.load("baseline"), p, settings=settings, score_from=start)
    return bt.equity_curve(pf, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0


panel = load(d.get("end"))          # TRAIN — what the study was fitted on
full = load(HOLDOUT_END)            # the whole history — every warm out-of-sample score
LAST = str(full.close.index[-1].date())
print(f"train {panel.close.index[0].date()} → {panel.close.index[-1].date()} "
      f"({len(panel.close):,} {panel.bar} bars, scored from {SCORE_FROM}) · "
      f"holdout {HOLDOUT_START} → {LAST}")

## 2 · Stress the candidates in-sample (train)

Everything here reads the training window only, so it can show a candidate is fragile, never that it is good. First each candidate against equal-weight buy-and-hold with the numbers the gates read: `alpha_cagr` is what is left after removing the market exposure `beta` explains; `excess_cagr` is the plain gap to a fully invested basket and is expected to be negative for a mostly-flat book.

In [ ]:
train = {label: score(p, panel, SCORE_FROM) for label, p in cands.items()}
COLS = {"cagr": PCT, "bench_cagr": PCT, "excess_cagr": PCT, "alpha_cagr": PCT, "beta": "{:+.2f}",
        "volatility": "{:.1%}", "bench_volatility": "{:.1%}", "sharpe": "{:.2f}",
        "max_drawdown": PCT, "pct_profitable": "{:.0%}", "n_trades": "{:.0f}",
        "trades_per_year": "{:.0f}"}
display(pd.DataFrame({label: {k: s[k] for k in COLS} for label, s in train.items()}).T
        .style.format(COLS))

### 2a · Costs, the pass bar and fill timing

Each candidate at 1×/2×/3× the configured costs, at the **pass bar** (10 bps spread + 3 bps slippage — the 2 bps default is a midday large-cap number and this strategy trades wallet-sized child orders in active names), the half-spread per leg at which its alpha reaches zero, and under `fill: pessimistic` (every buy at the bar's high, every sell at its low, benchmark included). Pessimistic fills are a bound, not a forecast: the truth lies between the two.

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})


def alpha(params, *, p=panel, start=SCORE_FROM, strategy=None, cfg=None):
    """Warm alpha after beta of one parameter set on one window."""
    return bt.evaluate(strategy or strat, p, params, settings=cfg or settings,
                       score_from=start)["alpha_cagr"]


def breakeven_bps(params):
    """Half-spread per leg (bps) at which the train alpha reaches zero (5 bps steps)."""
    base = settings.costs.spread_bps + settings.costs.slippage_bps
    for extra in np.arange(0, 65, 5.0):
        if alpha(params, cfg=with_costs(spread_bps=base + extra, slippage_bps=0.0)) <= 0:
            return base + extra
    return np.inf


c, worst = settings.costs, replace(strat, fill="pessimistic")
spread_bar, slip_bar = GATES["pass_bar_bps"]
costs = {}
for label, p in cands.items():
    row = {f"{m:g}x costs": alpha(p, cfg=with_costs(
        spread_bps=c.spread_bps * m, slippage_bps=c.slippage_bps * m,
        commission_per_share=c.commission_per_share * m)) for m in COST_MULTIPLES}
    row[f"pass bar {spread_bar:g}/{slip_bar:g}"] = alpha(
        p, cfg=with_costs(spread_bps=spread_bar, slippage_bps=slip_bar))
    row["breakeven bps/leg"] = breakeven_bps(p)
    row["pessimistic fills"] = alpha(p, strategy=worst)
    costs[label] = row
costs = pd.DataFrame(costs).T
display(costs.style.format(PCT).format("{:.0f}", subset=["breakeven bps/leg"]))

### 2b · Parameter stability — plateau or spike?

Each searched knob moved one at a time by ±5% and ±10% of its own searched span (log-aware, snapped to values the study could have suggested), re-scored on train. `spike` = both sides collapse: do not ship, prefer a flatter front row. The sweep behind this scores the whole panel cold on `excess_cagr`; the *relative* picture between a point and its neighbours is what matters, and it is the same for alpha up to the beta term.

In [ ]:
SWEEP_SETTINGS = settings.model_copy(
    update={"backtest": settings.backtest.model_copy(update={"chunk_size": 2})})
stability = {}
for label, p in cands.items():
    star = opt.sensitivity(strat, panel, p, settings=SWEEP_SETTINGS)
    verdicts = opt.stability_report(star, min_trades=FLOOR * train[label]["years"])
    center = float(star.loc[star["perturbed"] == "", "excess_cagr"].iloc[0])
    near = star.loc[star["perturbed"] != "", "excess_cagr"]
    stability[label] = {
        "neighbours": len(near), "spikes": int((verdicts["verdict"] == "spike").sum()),
        "cliffs": int(verdicts["verdict"].str.startswith("cliff").sum()),
        "inert": int((verdicts["verdict"] == "inert").sum()),
        "median neighbour − centre": float(near.median() - center)}
    if label == FOCUS:
        focus_star, focus_verdicts = star, verdicts
display(pd.DataFrame(stability).T.style.format("{:.0f}")
        .format(PCT, subset=["median neighbour − centre"]))
print(f"\n{FOCUS}, knob by knob:")
display(focus_verdicts)
plots.plot_sensitivity(focus_star).show()

### 2c · Month by month (train)

Each calendar month of the training window on its own, scored warm. An edge carried by one or two months is a regime bet, not a strategy — and for a strategy whose thesis is that the tape changes, this is where to look for *when* it worked.

In [ ]:
first_month = (pd.Timestamp(SCORE_FROM) + pd.offsets.MonthBegin(1)).normalize()
months = pd.date_range(first_month, pd.Timestamp(d["end"]), freq="MS")
monthly = {}
for m0 in months:
    m1 = min(m0 + pd.offsets.MonthEnd(1), pd.Timestamp(d["end"]))
    sub = slice_panel(panel, d["start"], str(m1.date()))          # history + the month
    monthly[m0.strftime("%Y-%m")] = {
        label: bt.evaluate(strat, sub, p, settings=settings,
                           score_from=str(m0.date()))["alpha_cagr"]
        for label, p in cands.items()}
month_table = pd.DataFrame(monthly).T.rename_axis("alpha per month (train, annualized)")
display(month_table.style.format(PCT).background_gradient(cmap="RdBu", vmin=-0.5, vmax=0.5))
print("months with positive alpha:",
      {label: f"{int((month_table[label] > 0).sum())}/{len(month_table)}" for label in cands})

### 2d · Trade anatomy (FOCUS)

Which rule ended each trade (the session-close flat, the time stop, the level stop, the target), how the realized loss per trade compares with `risk_per_trade` (a loser beyond it went *through* the stop — a gap or a fast market), the long/short split, and the time of day the entries happen. If one mechanism owns most exits, that mechanism *is* the strategy.

In [ ]:
focus = train[FOCUS]
pf_f, w_f = focus["pf"], focus["window"]
P = {**strat.params, **cands[FOCUS]}
trades = bt.trade_records(pf_f).copy()
idx = w_f.close.index
pos = pd.Series(np.arange(len(idx)), index=idx)
end_bar = bt.session_end(idx, w_f.tz)
mpb = strat.signals.__globals__["_minutes_per_bar"](w_f.bar)
horizon_b = int(np.ceil(P["horizon"] / mpb))
trades["bars_held"] = pos[pd.to_datetime(trades["exit_ts"])].to_numpy() - \
    pos[pd.to_datetime(trades["entry_ts"])].to_numpy()
trades["cause"] = np.select(
    [end_bar[pd.to_datetime(trades["exit_ts"])].to_numpy(), trades["bars_held"] >= horizon_b,
     trades["pnl"] > 0], ["session close", "time stop", "target"], "stop")
trades["loss_of_book"] = -trades["pnl"] / INIT_CASH
trades["hour"] = pd.to_datetime(trades["entry_ts"]).dt.tz_convert(w_f.tz).dt.hour
n = len(trades)
print(f"{FOCUS}: {n} trades on train (warm from {SCORE_FROM})")
win_rate = ("pnl", lambda s: (s > 0).mean())
display(trades.groupby("cause")
        .agg(trades=("pnl", "size"), share=("pnl", lambda s: len(s) / n), win_rate=win_rate,
             mean_pnl=("pnl", "mean"), median_bars=("bars_held", "median"))
        .sort_values("trades", ascending=False)
        .style.format({"share": "{:.0%}", "win_rate": "{:.0%}", "mean_pnl": "{:+.1f}"}))
losers = trades[trades["pnl"] < 0]
through = losers[losers["loss_of_book"] > P["risk_per_trade"] * 1.5]
print(f"losers: {len(losers)} · beyond 1.5x risk_per_trade ({P['risk_per_trade']:.2%} of the "
      f"book): {len(through)} ({len(through) / max(len(losers), 1):.0%}) — gaps through the stop")
risk_usd = P["risk_per_trade"] * INIT_CASH
display(trades.groupby("direction")
        .agg(trades=("pnl", "size"), win_rate=win_rate,
             mean_R=("pnl", lambda s: s.mean() / risk_usd))
        .style.format({"win_rate": "{:.0%}", "mean_R": "{:+.2f}"}))
display(trades.groupby("hour").agg(entries=("pnl", "size"), mean_pnl=("pnl", "mean"))
        .style.format({"mean_pnl": "{:+.1f}"}))

## 3 · The re-tuning procedure, out of sample

The thesis says no single parameter set works for months on end, so the procedure under test is **re-optimize the regime-sensitive knobs every six weeks on the last four months, then trade**. Each fold re-runs a real study (same space for the re-tuned knobs, same sampler, same objectives, same picking rule and floor as the optimization notebook) on `[train start, train end]`, **warm** — the strategy sees the whole history, the trial is scored only inside its window — then scores the fold's pick on the unseen six weeks that follow, warm. Fold studies are journals named by their **dates**, so a changed fold layout never silently resumes a study trained on another window.

Three arms on identical test windows:

- **refit** — each fold's own pick (the procedure);
- **frozen** — the first fold's pick, never re-tuned (the null: *tuning once was enough*);
- **defaults** — the shipped `config.yaml` values with the candidate's structural choices (the null: *tuning never helped*).

The candidate itself is also scored on each window, but its study *saw* these months: that row is in-sample and shows where its edge sits, never a pass/fail. Four folds is a thin sample: read the spread, not only the mean.

In [ ]:
def as_params(row, keys) -> dict:
    """A front row's searched parameters, cast back to the types signals() expects."""
    out = {}
    for k in keys:
        v = row[k]
        if isinstance(v, str):
            out[k] = v
        elif isinstance(v, (bool, np.bool_)):
            out[k] = bool(v)
        elif SPACE[k]["type"] == "categorical":
            out[k] = type(SPACE[k]["choices"][0])(v)
        elif k in SPACE_INT:
            out[k] = int(round(float(v)))
        else:
            out[k] = round(float(v), 10)
    return out


structural = {k: v for k, v in cands[FOCUS].items() if k in STRUCTURAL}
fold_strat = replace(strat, params={**strat.params, **structural},
                     optuna={**strat.optuna, "space": {k: SPACE[k] for k in RETUNE}})
defaults = {**{k: strat.params[k] for k in RETUNE}, **structural}
wf = []
for tr0, tr1, te0, te1 in FOLDS:
    name = f"{STUDY}-wf-{tr0}-{tr1}"
    try:
        done = len(opt.load_study(fold_strat, name).trials)
    except KeyError:
        done = 0
    fold_study = opt.run_study(fold_strat, slice_panel(panel, d["start"], tr1), settings=settings,
                               study_name=name, n_trials=max(0, WF_TRIALS - done), score_from=tr0)
    fold_front = opt.pareto_frame(fold_study)
    pool = fold_front[fold_front["trades_per_year"] >= FLOOR]
    row = opt.pick_from_front(pool if len(pool) else fold_front)
    picked = {**structural, **as_params(row, RETUNE)}
    window = slice_panel(panel, d["start"], te1)        # history + the test window, scored warm
    arms = {"refit": picked, "frozen": wf[0]["refit"] if wf else picked, "defaults": defaults,
            "candidate (in-sample)": cands[FOCUS]}
    oos = {arm: bt.evaluate(strat, window, prm, settings=settings, score_from=te0)
           for arm, prm in arms.items()}
    wf.append({"fold": f"{te0}..{te1}", "te0": te0, "te1": te1, "window": window,
               "IS alpha": float(row["alpha_cagr"]), "IS trades/yr": float(row["trades_per_year"]),
               **arms, **{f"{arm} OOS": st for arm, st in oos.items()},
               "scored": len([t for t in fold_study.trials if t.values is not None]),
               "beats": bool(row["beats_benchmark"])})
    print(f"fold {te0}..{te1}: {len(fold_study.trials)} trials ({wf[-1]['scored']} scored), "
          f"pick alpha {row['alpha_cagr']:+.1%} IS → OOS refit {oos['refit']['alpha_cagr']:+.1%} "
          f"· frozen {oos['frozen']['alpha_cagr']:+.1%} "
          f"· defaults {oos['defaults']['alpha_cagr']:+.1%}")

ARMS = ("refit", "frozen", "defaults", "candidate (in-sample)")
wf_table = pd.DataFrame({
    w_["fold"]: {"IS alpha (refit)": w_["IS alpha"],
                 **{f"{arm} alpha": w_[f"{arm} OOS"]["alpha_cagr"] for arm in ARMS},
                 **{f"{arm} trades": w_[f"{arm} OOS"]["n_trades"] for arm in ARMS[:3]},
                 "refit volatility": w_["refit OOS"]["volatility"]}
    for w_ in wf}).T
trade_cols = [c for c in wf_table if "trades" in c]
display(wf_table.style.format(PCT).format("{:.0f}", subset=trade_cols)
        .format("{:.1%}", subset=["refit volatility"]))
is_a = wf_table["IS alpha (refit)"].to_numpy()
oos_a = wf_table["refit alpha"].to_numpy()
WF_POSITIVE = int((oos_a > 0).sum())
t, p_value = stats.ttest_rel(is_a, oos_a)
print(f"refit: mean IS alpha {is_a.mean():+.2%}/yr vs mean OOS alpha {oos_a.mean():+.2%}/yr "
      f"(paired t={t:.2f}, p={p_value:.2f}) · positive OOS in {WF_POSITIVE}/{len(oos_a)} folds")
for arm in ("frozen", "defaults"):
    a = wf_table[f"{arm} alpha"].to_numpy()
    print(f"{arm:9s}: mean OOS alpha {a.mean():+.2%}/yr · positive in "
          f"{int((a > 0).sum())}/{len(a)}")
print("\nwhat each refit chose (agreement across folds = the knob is load-bearing):")
display(pd.DataFrame([{k: w_["refit"][k] for k in RETUNE} for w_ in wf],
                     index=wf_table.index).T)

### 3b · The stitched out-of-sample path

The four test windows chained into one path per arm — the closest thing to *"I had run this procedure live from November"*. Annualized return, volatility, drawdown, and alpha after beta against buy-and-hold over the same windows. This is the number the re-tuning gates read: the procedure must at least match the frozen control, or it is complexity without pay.

In [ ]:
def stitched(arm):
    """Chain the per-fold OOS per-bar returns of one arm into one path."""
    parts = []
    for w_ in wf:
        pf = bt.run_strategy(strat, w_["window"], w_[arm], settings=settings, score_from=w_["te0"])
        parts.append(bt.equity_curve(pf)["value"].pct_change().fillna(0.0))
    return pd.concat(parts)


bench_parts = []
for w_ in wf:
    test = slice_panel(w_["window"], w_["te0"], w_["te1"])
    fees, slip = bt.panel_costs(settings.costs, test)
    bench_parts.append(bt.benchmark(test, fees=fees, slippage=slip, fill=strat.fill)[1])
paths = {arm: stitched(arm) for arm in ARMS[:3]}
paths["buy & hold"] = pd.concat(bench_parts).reindex(paths["refit"].index).fillna(0.0)


def path_stats(r, bench):
    years = bt._years(r.index)
    total = float((1.0 + r).prod() - 1.0)
    curve = (1.0 + r).cumprod()
    var = float(bench.var())
    beta = float(r.cov(bench) / var) if var else 0.0
    cagr = bt._cagr(total, years)
    return {"cagr": cagr, "volatility": float(r.std() * np.sqrt(len(r) / years)),
            "max drawdown": float((curve / curve.cummax() - 1.0).min()), "beta": beta,
            "alpha_cagr": cagr - beta * bt._cagr(float((1.0 + bench).prod() - 1.0), years),
            "sharpe": float(r.mean() / r.std() * np.sqrt(len(r) / years)) if r.std() else 0.0}


stitch = pd.DataFrame({arm: path_stats(r, paths["buy & hold"]) for arm, r in paths.items()}).T
display(stitch.style.format({"cagr": PCT, "volatility": "{:.1%}", "max drawdown": PCT,
                             "beta": "{:+.2f}", "alpha_cagr": PCT, "sharpe": "{:.2f}"}))
fig = go.Figure()
for i, (arm, r) in enumerate(paths.items()):
    fig.add_trace(plots._scatter(
        (1.0 + r).cumprod() - 1.0, name=arm,
        line={"color": "#898781" if arm == "buy & hold" else plots.SERIES[i], "width": 1.5,
              "dash": "dash" if arm == "buy & hold" else "solid"},
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>" + arm + "</extra>"))
plots._base(fig, title="Stitched out-of-sample paths — refit vs frozen vs defaults, "
                       f"{len(wf)} folds", height=440)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()
print(f"weeks of out-of-sample path: {bt._years(paths['refit'].index) * 52:.0f}")

## 4 · The untouched holdout — open once

May–June 2026 (`HOLDOUT_START` → `HOLDOUT_END`, the last session every name has 1m bars for) was never loaded by the optimization notebook. Score the candidates on it **once**, warm, at default costs, at the pass bar and under pessimistic fills, and do not tune against what you see.

In [ ]:
hold, rows = {}, {}
worst = replace(strat, fill="pessimistic")
spread_bar, slip_bar = GATES["pass_bar_bps"]
for label, p in cands.items():
    hold[label] = h = score(p, full, HOLDOUT_START)
    h["pass_bar_alpha"] = alpha(p, p=full, start=HOLDOUT_START,
                                cfg=with_costs(spread_bps=spread_bar, slippage_bps=slip_bar))
    h["pessimistic_alpha"] = alpha(p, p=full, start=HOLDOUT_START, strategy=worst)
    rows[label] = {"CAGR": h["cagr"], "buy & hold": h["bench_cagr"], "excess": h["excess_cagr"],
                   "alpha": h["alpha_cagr"], "beta": h["beta"], "volatility": h["volatility"],
                   "B&H volatility": h["bench_volatility"], "max drawdown": h["max_drawdown"],
                   "trades": h["n_trades"],
                   f"alpha @ {spread_bar:g}/{slip_bar:g} bps": h["pass_bar_alpha"],
                   "alpha, pessimistic fills": h["pessimistic_alpha"]}
hold_table = pd.DataFrame(rows).T
display(hold_table.style.format(PCT).format("{:+.2f}", subset=["beta"])
        .format("{:.1%}", subset=["volatility", "B&H volatility"])
        .format("{:.0f}", subset=["trades"]))

fig = go.Figure()
fig.add_trace(plots._scatter(
    bench_curve(full, HOLDOUT_START), name="baseline (buy & hold)",
    line={"color": "#898781", "dash": "dash", "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>baseline</extra>"))
for i, (label, h) in enumerate(hold.items()):
    fig.add_trace(plots._scatter(
        bt.equity_curve(h["pf"], init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0, name=label,
        line={"color": plots.SERIES[i % len(plots.SERIES)], "width": 1.5},
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>" + label + "</extra>"))
plots._base(fig, title=f"Holdout {HOLDOUT_START} → {LAST} — cumulative return vs buy-and-hold",
            height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

## 5 · Verdict

One row per gate, one column per candidate. A **FAIL** on any hard gate fails the candidate; **WARN** is a soft gate and **INFO** is reported without a threshold. The walk-forward and stitched-path gates judge the *procedure*, so they read the same for every candidate; the holdout gates judge the candidate itself.

In [ ]:
def gate(ok, value, hard=True):
    """One scoreboard cell: PASS / FAIL (WARN for a soft gate) and the number behind it."""
    return f"{'PASS' if ok else 'FAIL' if hard else 'WARN'} · {value}"


wf_ok = bool(oos_a.mean() > 0) and GATES["wf_min_positive_folds"] <= WF_POSITIVE
refit_vs_frozen = stitch.loc["refit", "alpha_cagr"] - stitch.loc["frozen", "alpha_cagr"]
refit_vs_defaults = stitch.loc["refit", "alpha_cagr"] - stitch.loc["defaults", "alpha_cagr"]
pess_min = GATES["pessimistic_min_alpha"]
board = {}
for label in cands:
    tr, h = train[label], hold[label]
    cells_ = {
        "alpha · train": gate(tr["alpha_cagr"] > 0,
                              f"{tr['alpha_cagr']:+.1%} (beta {tr['beta']:+.2f})"),
        "alpha · holdout": gate(h["alpha_cagr"] > 0,
                                f"{h['alpha_cagr']:+.1%} (beta {h['beta']:+.2f})"),
        "walk-forward (procedure)": gate(wf_ok, f"{WF_POSITIVE}/{len(oos_a)} · "
                                                f"mean {oos_a.mean():+.1%}"),
        "pass bar · holdout": gate(h["pass_bar_alpha"] > 0, f"{h['pass_bar_alpha']:+.1%}"),
        "enough holdout trades": gate(h["n_trades"] >= GATES["min_holdout_trades"],
                                      f"{h['n_trades']:.0f}"),
        "no stability spike": gate(stability[label]["spikes"] == 0,
                                   f"{stability[label]['spikes']} spike(s)"),
        "re-tuning ≥ frozen (stitched)": gate(refit_vs_frozen >= 0, f"{refit_vs_frozen:+.1%}",
                                              hard=False),
        "re-tuning > defaults (stitched)": gate(refit_vs_defaults > 0, f"{refit_vs_defaults:+.1%}",
                                                hard=False),
        "volatility ≤ buy & hold · holdout": gate(
            h["volatility"] <= h["bench_volatility"],
            f"{h['volatility']:.1%} vs {h['bench_volatility']:.1%}", hard=GATES["vol_is_hard"]),
        "pessimistic fills": (f"INFO · {h['pessimistic_alpha']:+.1%}" if pess_min is None
                              else gate(h["pessimistic_alpha"] > pess_min,
                                        f"{h['pessimistic_alpha']:+.1%}")),
    }
    failed = [k for k, v in cells_.items() if v.startswith("FAIL")]
    cells_["VERDICT"] = "PASS" if not failed else f"FAIL ({len(failed)} gate(s))"
    board[label] = cells_

scoreboard = pd.DataFrame(board)
COLOR = {"PASS": "#d8efd6", "FAIL": "#f6d5d1", "WARN": "#fbe8c6", "INFO": ""}
display(scoreboard.style.map(lambda v: f"background: {COLOR.get(v.split(' ')[0], '')}"))
print("ship frozen" if refit_vs_frozen < 0 else "ship with re-tuning",
      "— the procedure gate says so; the holdout gates say whether to ship at all.")

## 6 · Promote a candidate into `config.yaml`

Off by default: this **rewrites a tracked file**, and what ships is a decision, not a side effect of running a notebook. It refuses a candidate whose verdict failed unless `FORCE = True` (say why in the commit). The rewrite is surgical — only the `value:` lines change, every comment survives — and it verifies itself by reloading the strategy, restoring the original file if the config no longer loads or a value does not round-trip.

In [ ]:
import pathlib
import re

PROMOTE = FOCUS        # which candidate to write into config.yaml
WRITE_BACK = False     # True rewrites strategies/level_flow/config.yaml
FORCE = False          # True writes a candidate that failed the verdict


def _yaml_scalar(v):
    """Render one value the way config.yaml writes it."""
    if isinstance(v, bool):
        return "true" if v else "false"
    return f'"{v}"' if isinstance(v, str) else repr(v)


def promote(params, *, apply=False):
    """Rewrite `params.<key>.value` for every knob that differs; verify by reloading."""
    path = pathlib.Path(strat.dir) / "config.yaml"
    original = path.read_text()
    cfg = yaml.safe_load(original)
    lines, changes = original.splitlines(), []
    for key, new in params.items():
        spec = cfg["params"].get(key)
        if not isinstance(spec, dict) or spec.get("value") == new:
            continue
        at = next((i for i, ln in enumerate(lines) if re.fullmatch(rf"  {key}:\s*", ln)), None)
        if at is None:
            print(f"  ! {key}: no long-form entry to write to — edit it by hand")
            continue
        end = next((i for i in range(at + 1, len(lines))
                    if lines[i][:3].strip() and not lines[i].startswith("    ")), len(lines))
        for i in range(at + 1, end):
            if lines[i].split("#")[0].strip().startswith("value:"):
                note = lines[i].partition("#")[2]
                lines[i] = f"    value: {_yaml_scalar(new)}" + (f"   #{note}" if note else "")
        changes.append((key, spec.get("value"), new))
    for key, old, new in changes:
        print(f"  {key:18s} {old!r:>12} -> {new!r}")
    if not changes:
        print("config.yaml already ships these values — nothing to promote.")
        return changes
    if not apply:
        print(f"\n{len(changes)} knob(s) would change. Set WRITE_BACK = True to apply.")
        return changes
    path.write_text("\n".join(lines) + "\n")
    try:
        reloaded = strategies.load(strat.name)            # the config invariants run here
    except Exception:
        path.write_text(original)
        print("REVERTED — the rewritten config does not load:")
        raise
    if bad := {k: v for k, v in params.items() if reloaded.params.get(k) != v}:
        path.write_text(original)
        raise AssertionError(f"REVERTED — values did not round-trip: {bad}")
    print(f"\nwrote {path} — commit the diff, then re-run both notebooks from the top.")
    return changes


verdict = scoreboard.loc["VERDICT", PROMOTE]
print(f"{PROMOTE}: verdict {verdict}\n")
if WRITE_BACK and verdict != "PASS" and not FORCE:
    raise RuntimeError(f"{PROMOTE} failed the verdict — set FORCE = True to write it anyway")
_ = promote(cands[PROMOTE], apply=WRITE_BACK)

## Read the result honestly

- The **walk-forward** and the **holdout** are the real numbers; §2 can only show fragility.
- The walk-forward judges the **procedure** (re-tune, freeze, or never tune); only the holdout judges a **candidate**. A candidate needs both.
- Comparing many candidates on the holdout and shipping the winner is selection on the holdout: pick among candidates on §2 and the walk-forward, then read the holdout as confirmation.
- Ten months of train and two of holdout is a short sample on one regime. A ~6-month stitched out-of-sample path cannot establish a Sharpe with any confidence (the minimum track record for a Sharpe of 1 at 95% is years, not months); read consistency across folds and months, not the headline.
- `excess_cagr` against a fully invested basket is negative by construction here; alpha after beta is the question — and if alpha is positive only at 2 bps and dies at the 10/3 pass bar, the edge is smaller than the spread it must cross.
- The pre-registered rules in the README decide what ships. Anything less ships as a finding, not a config.